In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)

# Fresh load - untouched raw data
df = pd.read_csv('../data/raw/india_job_market_2025.csv')

print(f"Starting shape: {df.shape}")
# Expected: (97929, 17)

Starting shape: (97929, 17)


In [2]:
# Identify rows missing all 5 critical columns simultaneously
critical_cols = ['minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience', 'tagsAndSkills']
systemic_failure_mask = df[critical_cols].isnull().all(axis=1)

print(f"Rows to drop (systemic failure): {systemic_failure_mask.sum()}")

df = df[~systemic_failure_mask].copy()

print(f"Shape after dropping systemic-failure rows: {df.shape}")
# Expected: (97358, 17)

Rows to drop (systemic failure): 571
Shape after dropping systemic-failure rows: (97358, 17)


In [3]:
full_dupes_count = df.duplicated().sum()
print(f"Full duplicate rows to drop: {full_dupes_count}")

df = df.drop_duplicates(keep='first').copy()

print(f"Shape after dropping full duplicates: {df.shape}")
# Expected: (97111, 17)

Full duplicate rows to drop: 247
Shape after dropping full duplicates: (97111, 17)


In [4]:
# Drop rows where BOTH jobId AND title match (true duplicates)
# This correctly spares the 11025020549 pair since their titles differ
true_dupe_mask = df.duplicated(subset=['jobId', 'title'], keep='first')

print(f"True partial duplicates to drop: {true_dupe_mask.sum()}")

df = df[~true_dupe_mask].copy()

print(f"Shape after resolving partial duplicates: {df.shape}")
# Expected: (97109, 17)

True partial duplicates to drop: 2
Shape after resolving partial duplicates: (97109, 17)


In [5]:
missing_company = df['companyName'].isnull().sum()
print(f"Rows missing companyName: {missing_company}")

df = df.dropna(subset=['companyName']).copy()

print(f"Shape after dropping missing companyName: {df.shape}")
# Expected: (97105, 17)

Rows missing companyName: 4
Shape after dropping missing companyName: (97105, 17)


In [6]:
print("=" * 60)
print("PHASE 5 - CHECKPOINT 1 SUMMARY")
print("=" * 60)
print(f"Original rows:        97,929")
print(f"Current rows:          {len(df):,}")
print(f"Total removed:         {97929 - len(df):,}")
print(f"Removal percentage:    {(97929 - len(df)) / 97929 * 100:.2f}%")
print(f"\nRemaining null check:")
print(df[critical_cols].isnull().sum())

PHASE 5 - CHECKPOINT 1 SUMMARY
Original rows:        97,929
Current rows:          97,105
Total removed:         824
Removal percentage:    0.84%

Remaining null check:
minimumSalary        0
maximumSalary        0
minimumExperience    0
maximumExperience    0
tagsAndSkills        0
dtype: int64


In [7]:
import re

def extract_work_mode(location_str):
    """
    Extracts work mode (Remote/Hybrid/Onsite) from the location string.
    Returns the work mode and does NOT modify the original string.
    """
    location_lower = str(location_str).lower()
    
    if 'remote' in location_lower:
        return 'Remote'
    elif 'hybrid' in location_lower:
        return 'Hybrid'
    else:
        return 'Onsite'

df['work_mode'] = df['location'].apply(extract_work_mode)

print(df['work_mode'].value_counts())
print(f"\nTotal: {df['work_mode'].value_counts().sum()}")

work_mode
Onsite    89452
Hybrid     5733
Remote     1920
Name: count, dtype: int64

Total: 97105


In [ ]:
# Rows that were originally exactly "Remote" but did NOT get classified as Remote now
exactly_remote = df[df['location'].str.strip().str.lower() == 'remote']
print(f"Rows where location is exactly 'Remote': {len(exactly_remote)}")

not_classified_remote = exactly_remote[exactly_remote['work_mode'] != 'Remote']
print(f"Of those, how many were NOT classified as Remote: {len(not_classified_remote)}")
print(not_classified_remote[['location', 'work_mode']].head())